# Rotation Correlation Curve Analysis

Standalone debug viewer for the **2D SOFT rotation correlation** (all angles in **radians**). It reads the CSV dumps that the fs2d debug run already produces (no C++ changes needed).

## Workflow
1. Run a registration with the debug flag on (service or test executable, `useDirect=true` for the coefficient files).
2. In this notebook click **Run All**.
3. Zoom into any region of the curve (rangeslider at the bottom, or the lo/hi sliders) to inspect its curvature.
4. The last section fits the curve with the **true kernel** and runs a **hidden-component scan** that can reveal *shoulder/plateau* features which classical peak detection cannot see (e.g. the ~0.55 rad / GT region of the earlier pair 285->290).

## Files read from `./data`
| file | content |
|---|---|
| `rotationCorrelation1D.csv` | correlation curve `[index, angle(rad), normalizedCorrelation]` |
| `rotationPeaks.csv` | detected peaks `[angle, peakCorrelation, covariance, levelPotential, index]` |
| `registration_meta.csv` | pair + estimated rotation + GT error (one row) |
| `dataForReadIn.csv` | parameters (N, thresholds, numAngles, numTotalSolutions) |
| `sigCoefR/I_1angle.csv`, `patCoefR/I_1angle.csv` | spherical-harmonic coefficients (written by the `useDirect=true` debug path) -> true kernel |

Requirements: `numpy`, `plotly`, `ipywidgets` (same env as `plot_2d_registration.ipynb`).


In [63]:
"""Imports + data location."""
import os
import numpy as np
import ipywidgets as widgets
from IPython.display import display, Markdown

# Data directory: this notebook lives in plotting_results/2d, debug output in ./data
DATA_DIR = os.path.join(os.getcwd(), "data")
if not os.path.isdir(DATA_DIR):
    DATA_DIR = "/home/tim-external/ros_ws/src/fsregistration/plotting_results/2d/data"
print("DATA_DIR:", DATA_DIR)


DATA_DIR: /home/tim-external/ros_ws/src/fsregistration/plotting_results/2d/data


In [64]:
"""Robust loaders (tolerate headers, tab/comma/space delimiters, trailing non-numeric tokens)."""

def load_lines(fname):
    """All parsed rows (list of lists of floats); headers / non-numeric rows skipped."""
    path = os.path.join(DATA_DIR, fname)
    if not os.path.isfile(path):
        return None
    out = []
    with open(path) as f:
        for ln in f:
            ln = ln.strip()
            if not ln or ln.startswith("#"):
                continue
            vals = []
            for t in ln.replace(",", " ").split():
                try:
                    vals.append(float(t))
                except ValueError:
                    break
            if vals:
                out.append(vals)
    return out

def load_csv(fname):
    """2D array using the dominant column count (e.g. registration_meta keeps its 13 numeric cols)."""
    rows = load_lines(fname)
    if not rows:
        return None
    counts = [len(r) for r in rows]
    modal = max(set(counts), key=counts.count)
    rows = [r for r in rows if len(r) == modal]
    return np.array(rows) if rows else None

def status(name, arr):
    print("  [%s] %s%s" % ("ok" if arr is not None else "MISSING", name,
                           "  (%d rows)" % arr.shape[0] if arr is not None and arr.ndim == 2 else ""))


In [65]:
"""Load everything and print a summary (angles in rad)."""
print("Loading debug files from", DATA_DIR)
curve = load_csv("rotationCorrelation1D.csv")   # [index, angle(rad), normalizedCorrelation]
peaks = load_csv("rotationPeaks.csv")           # [angle, peakCorrelation, covariance, levelPotential, index]
meta  = load_csv("registration_meta.csv")       # frame1 frame2 rot_angle_deg tx ty ...
cfg   = load_lines("dataForReadIn.csv")         # parameter rows of mixed width

status("rotationCorrelation1D.csv", curve)
status("rotationPeaks.csv", peaks)
status("registration_meta.csv", meta)
import time as _t
for _f in ("rotationCorrelation1D.csv", "rotationPeaks.csv", "registration_meta.csv"):
    _p = os.path.join(DATA_DIR, _f)
    if os.path.isfile(_p):
        print("    mtime %s: %s" % (_f, _t.strftime("%Y-%m-%d %H:%M:%S", _t.localtime(os.path.getmtime(_p)))))

# --- parameters (dataForReadIn: 6-col metadata row + per-angle 3-col rows) ---
N = num_angles = num_total = None
level_thresh = None
if cfg:
    for r in cfg:
        if len(r) == 6:
            N = int(r[0]); _cn = int(r[1]); _cell = r[2]
            level_thresh = r[3]; num_angles = int(r[4]); num_total = int(r[5])
            break
print("\nParameters: N=%s  level_potential=%s  numAngles=%s  numTotalSolutions=%s" % (N, level_thresh, num_angles, num_total))

# --- ground truth (derived: GT = estimated rotation + GT error), converted to rad ---
gt_rad, pair = None, None
if meta is not None and len(meta) >= 1:
    pair = (int(meta[0, 0]), int(meta[0, 1]))
    est_deg = float(meta[0, 2])
    gt_err_deg = float(meta[0, 7]) if meta.shape[1] > 7 else 0.0
    gt_rad = np.deg2rad(est_deg + gt_err_deg)
    print("Pair: %d -> %d   estimated rotation: %.4f rad   GT error: %.4f rad   -> GT: %.4f rad"
          % (pair[0], pair[1], np.deg2rad(est_deg), np.deg2rad(gt_err_deg), gt_rad))
else:
    print("No registration_meta.csv -> ground truth marker will not be shown.")

# --- detected peaks ---
if peaks is not None:
    print("\nDetected rotation peaks (from rotationPeaks.csv):")
    print("  %10s %11s %13s" % ("angle(rad)", "correlation", "levelPotential"))
    for p in peaks:
        print("  %10.4f %11.4f %13.4f" % (p[0], p[1], p[3]))
    print("  Note: every peak has an antipodal copy at +pi rad (the curve is pi-periodic).")
    print("  Physical rotations are angles modulo pi rad.")


Loading debug files from /home/tim-external/ros_ws/src/fsregistration/plotting_results/2d/data
  [ok] rotationCorrelation1D.csv  (4096 rows)
  [ok] rotationPeaks.csv  (6 rows)
  [ok] registration_meta.csv  (1 rows)
    mtime rotationCorrelation1D.csv: 2026-08-27 11:05:28
    mtime rotationPeaks.csv: 2026-08-27 11:05:28
    mtime registration_meta.csv: 2026-08-27 11:20:20

Parameters: N=256  level_potential=0.01  numAngles=6  numTotalSolutions=151
Pair: 0 -> 5   estimated rotation: 5.1083 rad   GT error: 0.1277 rad   -> GT: 5.2360 rad

Detected rotation peaks (from rotationPeaks.csv):
  angle(rad) correlation levelPotential
      0.4310      0.8610        0.7414
      0.9557      0.3922        0.1538
      2.0739      1.0000        1.0000
      3.5726      0.8610        0.7413
      4.0973      0.3922        0.1538
      5.2155      1.0000        1.0000
  Note: every peak has an antipodal copy at +pi rad (the curve is pi-periodic).
  Physical rotations are angles modulo pi rad.


In [66]:
"""Build the (interactive) correlation curve figure."""
import plotly.graph_objects as go

fig1 = go.Figure()   # plain Figure: no anywidget dependency
if curve is None:
    print("No correlation curve available.")
else:
    x, c = curve[:, 1], curve[:, 2]

    # robust fold: uniform [0, 2pi) grid with an even number of samples
    n = len(x)
    step = np.median(np.diff(x))
    uniform = np.allclose(np.diff(x), step, atol=1e-9)
    if not uniform or not np.isclose(x[0], 0.0, atol=1e-6):
        xg = np.linspace(0.0, 2 * np.pi, n)
        c = np.interp(xg, x, c)
        x = xg
    n = len(x) - (len(x) % 2)
    x, c = x[:n], c[:n]
    half = n // 2
    fold_x = x[:half]
    fold_c = (c[:half] + c[half:]) / 2.0

    fig1.add_trace(go.Scatter(x=x, y=c, name="correlation C(theta)",
                              line=dict(color="steelblue", width=2),
                              hovertemplate="%{x:.4f} rad<br>corr %{y:.4f}<extra></extra>"))
    fig1.add_trace(go.Scatter(x=fold_x, y=fold_c, name="folded [0,pi) = (C+C(pi))/2",
                              line=dict(color="orange", width=1.5, dash="dot"), visible="legendonly"))

    if peaks is not None:
        fig1.add_trace(go.Scatter(x=peaks[:, 0], y=peaks[:, 1], mode="markers+text",
                                  name="detected peaks",
                                  marker=dict(symbol="x", size=12, color="red", line=dict(width=2)),
                                  text=["%.4f rad" % a for a in peaks[:, 0]],
                                  textposition="top center", textfont=dict(size=10, color="red")))

    if gt_rad is not None:
        fig1.add_vline(x=gt_rad, line=dict(color="green", dash="dash", width=1.5),
                       annotation_text="GT %.4f rad" % gt_rad, annotation_position="top left")

    cm = c.min() if len(c) else 0.0
    fig1.update_layout(
        title="Rotation correlation curve, pair %d -> %d" % (pair[0], pair[1]) if pair else "Rotation correlation curve",
        height=520,
        legend=dict(orientation="h", y=1.12, font=dict(size=11)),
        xaxis=dict(title="rotation angle (rad)", rangeslider=dict(visible=True),
                   range=[0, 2 * np.pi], showgrid=True,
                   tickvals=[0, np.pi / 2, np.pi, 3 * np.pi / 2, 2 * np.pi],
                   ticktext=["0", "pi/2", "pi", "3pi/2", "2pi"]),
        yaxis=dict(title="normalized correlation", range=[min(cm, 0) - 0.05, 1.05]),
        margin=dict(t=80))
    print("Tip: drag the rangeslider at the bottom to zoom, or use the sliders below.")


Tip: drag the rangeslider at the bottom to zoom, or use the sliders below.


In [67]:
"""Zoom helpers: set the visible x-range of the figure above (re-rendered via an Output widget)."""
lo = widgets.FloatSlider(value=0.0, min=0.0, max=2 * np.pi, step=0.01, description="lo (rad)")
hi = widgets.FloatSlider(value=2 * np.pi, min=0.0, max=2 * np.pi, step=0.01, description="hi (rad)")
btn = widgets.Button(description="Apply zoom")
reset = widgets.Button(description="Reset")
out = widgets.Output()

def _redraw():
    with out:
        out.clear_output(wait=True)
        display(fig1)

def _apply(_b):
    a, b = sorted([lo.value, hi.value])
    fig1.update_xaxes(range=[a, b])
    _redraw()

def _reset(_b):
    fig1.update_xaxes(range=[0, 2 * np.pi])
    _redraw()

btn.on_click(_apply)
reset.on_click(_reset)
_redraw()
display(widgets.VBox([widgets.HBox([lo, hi]), widgets.HBox([btn, reset]), out]))


## Kernel fit & hidden-component scan

**What is the "true kernel"?** A matched rotation peak does not look Gaussian in this pipeline: its shape is the autocorrelation of the resampled reference descriptor (a band-limited bump with tails), computable exactly and for free from the SH coefficients of the debug dump:

$$K(\theta) = \sum_{m \neq 0} \left(\sum_{\ell} |\hat b_{\ell m}|^2\right) \cos(m\theta)$$

**Why fold?** The curve is (nearly) exactly pi-periodic: $C(\theta+\pi)=C(\theta)$. Everything lives in $[0, \pi)$; antipodal copies need no special handling.

**The scan (GT-free):** classical persistence detection only supplies the *initial hypotheses*. The kernel fit then probes a window around **every** persistence peak ($\pm$ `WIN_HALF_RAD`) for ONE additional component - a *shoulder/plateau* that is not a local maximum and is invisible to classical peak detection (e.g. the ~0.55 rad plateau of pair 285->290, which was the true rotation yet invisible). A candidate counts only if it survives the non-negative fit AND improves the residual by >= `MIN_IMPROV_RATIO`; the strongest accepted candidate becomes the hidden plateau. GT is only read from the meta row for a final "did we pick correctly" line - it never steers the scan.

Tune `WIN_HALF_RAD`, `WIN_CENTER_RAD` (set a number to force a single window), `MIN_SEP_RAD`, `KNOWN_MARGIN_RAD` in the next cell if needed.


In [68]:
"""True kernel from the SH coefficients of the reference (scan 2) descriptor + scan parameters (in rad)."""
# --- fit parameters (all in radians) --------------------------------
COARSE_RAD = 0.01        # hidden-component scan grid step (~0.57 deg)
FINE_RAD = 0.0004        # local refinement step (~0.02 deg)
MIN_SEP_RAD = 0.1        # min separation between components (~5.7 deg)
WIN_CENTER_RAD = None    # None = GT-free per-peak scan (window around EVERY persistence peak)
                         # set a number (e.g. GT mod pi) to force a single window (debug)
WIN_HALF_RAD = 0.35      # scan window half width (~20 deg)
MIN_IMPROV_RATIO = 2.0   # min residual improvement for a hidden candidate to be accepted
KNOWN_MARGIN_RAD = 0.26  # persistence peaks within +/-this of the window are known components

c2R = load_csv("patCoefR_1angle.csv")
c2I = load_csv("patCoefI_1angle.csv")
c1R = load_csv("sigCoefR_1angle.csv")
c1I = load_csv("sigCoefI_1angle.csv")

def build_kernel_acf(cR, cI, theta):
    """Autocorrelation kernel of a descriptor from its SH coefficients:
    K(theta) = sum_{m!=0} (sum_l |c_lm|^2) cos(m theta).
    The coefficient array uses the alm layout of softRegistrationClass.cpp
    (see the almIdx formula there); the file is a raw dump of that array."""
    bw = int(round(np.sqrt(len(cR))))
    bigL = bw - 1
    Q = np.zeros(2 * bw)
    for l in range(bw):
        for m in range(-l, l + 1):
            if m >= 0:
                idx = m * (bigL + 1) - m * (m - 1) // 2 + (l - m)
            else:
                idx = bigL * (bigL + 3) // 2 + 1 + (bigL + m) * (bigL + m + 1) // 2 + (l - abs(m))
            Q[m + bw] += cR[idx] ** 2 + cI[idx] ** 2
    mpos = np.arange(1, bw)                       # Q is symmetric in m (descriptor is real)
    K = 2.0 * (Q[mpos + bw] @ np.cos(np.outer(mpos, theta)))
    return K

def build_kernel_empirical(theta):
    """Fallback: measure the peak shape from the strongest peak of the observed
    folded curve itself (mirrored, 0.9 rad wide)."""
    Fk = fold_c
    i0 = int(np.argmax(Fk))
    W = int(round(0.9 / (theta[1] - theta[0])))
    seg = Fk[max(0, i0 - W):i0 + W + 1]
    K = np.zeros(len(theta))
    for j in range(W + 1):
        v = (seg[W - j] + (seg[W + j] if W + j < len(seg) else 0.0)) / 2.0
        K[j] = v
        if j > 0:
            K[len(theta) - j - 1] = v
    return K

if curve is None:
    print("Kernel panel skipped: no correlation curve.")
    K = None
elif c2R is not None and c2I is not None:
    print("Building true kernel K from scan-2 descriptor (patCoef, %d coefficients)" % len(c2R))
    K = build_kernel_acf(c2R[:, 0], c2I[:, 0], x)
    K = (K - K.min()) / (K.max() - K.min())
elif c1R is not None and c1I is not None:
    print("WARNING: patCoef missing -> using scan-1 descriptor (sigCoef) as kernel.")
    K = build_kernel_acf(c1R[:, 0], c1I[:, 0], x)
    K = (K - K.min()) / (K.max() - K.min())
else:
    print("WARNING: no coefficient files -> measuring the kernel empirically from the dominant peak.")
    K = build_kernel_empirical(x)
    K = (K - K.min()) / (K.max() - K.min())


Building true kernel K from scan-2 descriptor (patCoef, 16384 coefficients)


In [69]:
"""GT-free hidden-component scan: for EVERY persistence peak (folded mod pi),
scan its window (peak +- WIN_HALF_RAD) for ONE additional kernel component.

Classical persistence detection supplies the initial rotation hypotheses; the
kernel fit then tests each neighborhood. A candidate counts only if it keeps a
positive amplitude AND improves the residual by >= MIN_IMPROV_RATIO. The
strongest accepted candidate becomes the hidden plateau. GT (meta row) is only
used for a final validation line, never to center a window."""
mus_known, amps_known = [], []
hidden_result = None
kernel_peaks = []          # per fitted component: mu, amp, hidden?, shoulder?, anchor
scan_rad, scan_resid = [], []
peak_scan = []             # per persistence peak: window scan result dict

def circ_dist(a, b, period=np.pi):
    d = np.abs(np.asarray(a) - b) % period
    return np.minimum(d, period - d)

def kval(a):
    w = np.mod(np.asarray(a), 2 * np.pi)
    return np.interp(np.minimum(w, 2 * np.pi - w), x, K)   # K is even on [0, 2pi)

def design_matrix(angles, th):
    return np.column_stack([kval(th - a) for a in angles] + [np.ones(len(th))])

def fit_nonneg(th, Fw, angles_in):
    """Least squares with non-negative component amplitudes, affine baseline.
    Returns (coef, r, kept_angles); negative-amplitude components are dropped."""
    kept = list(angles_in)
    while True:
        A = design_matrix(kept, th)
        coef, res, *_ = np.linalg.lstsq(A, Fw, rcond=None)
        r = res[0] if len(res) else float(np.sum((A @ coef - Fw) ** 2))
        amps = coef[:-1]
        if len(amps) == 0 or np.all(amps >= 0):
            return coef, r, kept
        del kept[int(np.argmin(amps))]

def scan_window(ctr, th, F, known_all):
    """Scan ONE window centered at ctr (rad, folded circle). Returns a result
    dict or None when the window is empty / holds no valid candidate.
    The window is clamped to [0, pi] and candidates must stay inside it."""
    lo, hi = max(ctr - WIN_HALF_RAD, 0.0), min(ctr + WIN_HALF_RAD, np.pi)
    win = (th >= lo) & (th <= hi)
    thw, Fw = th[win], F[win]
    if len(thw) < 100:
        return None
    win_samp = np.arange(lo, hi, 0.01)              # window on the folded circle
    m0 = [m for m in known_all if np.min(circ_dist(m, win_samp)) < KNOWN_MARGIN_RAD]
    m0 = list(dict.fromkeys(m0))
    coef_k, r_k, kept_k = fit_nonneg(thw, Fw, m0)
    cand = np.arange(lo + COARSE_RAD, hi - COARSE_RAD, COARSE_RAD)
    resids = []
    for mu in cand:
        if any(circ_dist(mu, m) < MIN_SEP_RAD for m in kept_k):
            resids.append(np.inf)
            continue
        coef, r, kept = fit_nonneg(thw, Fw, kept_k + [mu])
        resids.append(r if kept == kept_k + [mu] else np.inf)
    if len(cand) == 0 or np.isinf(resids).all():
        return None
    resids = np.array(resids)
    ibest = int(np.argmin(resids))
    mu0, r0 = cand[ibest], float(resids[ibest])
    lo_c, hi_c = cand[0], cand[-1]                  # refinement stays in the window
    for mu_c in np.arange(max(mu0 - 0.044, lo_c), min(mu0 + 0.044, hi_c), FINE_RAD):
        if any(circ_dist(mu_c, m) < MIN_SEP_RAD for m in kept_k):
            continue
        coef, r, kept = fit_nonneg(thw, Fw, kept_k + [mu_c])
        if kept != kept_k + [mu_c]:
            continue
        if r < r0:
            r0, mu0 = r, float(mu_c)
    coef, r, kept = fit_nonneg(thw, Fw, kept_k + [mu0])
    lm = [th[i] for i in range(1, len(th) - 1) if F[i] > F[i - 1] and F[i] > F[i + 1]]
    return dict(center=float(ctr), lo=lo, hi=hi,
                mu=float(mu0), amp=float(coef[len(kept_k)]),
                resid0=r_k, resid1=r, ratio=r_k / r,
                shoulder=not any(circ_dist(mu0, m) < 0.026 for m in lm),
                coef=coef, mus_all=kept, knowns=kept_k,
                cand=cand, resids=resids)

if K is not None:
    th, F = fold_x, fold_c                         # folded grid [0, pi)
    known_all = []
    if peaks is not None:
        for p in peaks[:, 0]:
            m = float(p) % np.pi
            if not any(circ_dist(m, u) < 2e-4 for u in known_all):
                known_all.append(m)
        known_all.sort()
    print("Persistence peaks on [0, pi): %s" % [round(m, 4) for m in known_all])

    if WIN_CENTER_RAD is not None:
        centers = [float(WIN_CENTER_RAD % np.pi)]
        print("Scan mode: SINGLE window at %.4f rad (debug override WIN_CENTER_RAD)" % centers[0])
    else:
        centers = list(known_all)                   # GT-free: one window per persistence peak
        print("Scan mode: GT-FREE per-peak scan over %d window(s)" % len(centers))

    for ctr in centers:
        r = scan_window(ctr, th, F, known_all)
        if r is None:
            print("  window at %.4f rad: empty / no valid candidate" % ctr)
            continue
        r["accepted"] = r["ratio"] >= MIN_IMPROV_RATIO
        peak_scan.append(r)
        flag = "" if r["accepted"] else "  (below %.1fx threshold -> rejected)" % MIN_IMPROV_RATIO
        print("  anchor %.4f rad, window [%.4f, %.4f]: best hidden mu = %.4f, A = %.3f,"
              " resid %.4f -> %.4f (%.1fx)%s"
              % (ctr, r["lo"], r["hi"], r["mu"], r["amp"], r["resid0"], r["resid1"], r["ratio"], flag))

    found = [r for r in peak_scan if r["accepted"]]
    if found:
        hidden_result = max(found, key=lambda r: r["ratio"])
        h = hidden_result
        mus_known = h["knowns"]
        amps_known = [float(a) for a in h["coef"][:len(mus_known)]]
        scan_rad, scan_resid = list(h["cand"]), list(h["resids"])
        lm = [th[i] for i in range(1, len(th) - 1) if F[i] > F[i - 1] and F[i] > F[i + 1]]
        kernel_peaks = []
        for _j, _m in enumerate(h["mus_all"]):
            _h = (_j == len(mus_known))
            _s = not any(circ_dist(_m, _L) < 0.026 for _L in lm)
            kernel_peaks.append(dict(mu=float(_m), amp=float(h["coef"][_j]),
                                     hidden=bool(_h), shoulder=bool(_s),
                                     anchor=float(h["center"])))
        print("\nRESULT: hidden plateau at %.4f rad, A = %.3f, residual %.4f -> %.4f (%.1fx)"
              % (h["mu"], h["amp"], h["resid0"], h["resid1"], h["ratio"]))
        print("  window [%.4f, %.4f] rad around anchor %.4f rad"
              % (h["lo"], h["hi"], h["center"]))
        print("  -> %s" % ("SHOULDER/PLATEAU (invisible to maxima-based peak detection)"
                           if h["shoulder"] else "local maximum (classical detection would find it)"))
    else:
        hidden_result = None
        print("\nNo hidden plateau accepted: no window improved the residual by >= %.1fx."
              % MIN_IMPROV_RATIO)
else:
    print("Kernel fit skipped (K not available, see previous cell).")


Persistence peaks on [0, pi): [0.431, 0.9557, 2.0739]
Scan mode: GT-FREE per-peak scan over 3 window(s)
  anchor 0.4310 rad, window [0.0810, 0.7810]: best hidden mu = 0.5462, A = 0.204, resid 1.7428 -> 0.1829 (9.5x)
  anchor 0.9557 rad, window [0.6057, 1.3057]: best hidden mu = 0.6157, A = 0.099, resid 0.3185 -> 0.1752 (1.8x)  (below 2.0x threshold -> rejected)
  anchor 2.0739 rad, window [1.7239, 2.4239]: best hidden mu = 1.7339, A = 0.070, resid 0.4875 -> 0.3262 (1.5x)  (below 2.0x threshold -> rejected)

RESULT: hidden plateau at 0.5462 rad, A = 0.204, residual 1.7428 -> 0.1829 (9.5x)
  window [0.0810, 0.7810] rad around anchor 0.4310 rad
  -> SHOULDER/PLATEAU (invisible to maxima-based peak detection)


In [70]:
"""Figure 2: folded curve, scan window, kernel model, components, residual (rad)."""
import plotly.graph_objects as go

if K is not None and hidden_result is not None:
    thd = th                                        # folded grid in rad
    mus_all = hidden_result["mus_all"]
    coef_all = hidden_result["coef"]
    model_all = design_matrix(mus_all, th) @ coef_all

    fig2 = go.Figure()
    fig2.add_vrect(x0=hidden_result["lo"], x1=hidden_result["hi"],
                   fillcolor="lightgray", opacity=0.25, line_width=0,
                   annotation_text="scan window (anchor %.4f rad)" % hidden_result["center"],
                   annotation_position="top left")
    fig2.add_trace(go.Scatter(x=thd, y=F, name="folded observed", line=dict(color="black", width=2)))
    fig2.add_trace(go.Scatter(x=thd, y=model_all, name="kernel model", line=dict(color="steelblue", width=1.8)))
    for j, mu in enumerate(mus_all):
        is_hidden = j >= len(mus_known)
        col = "darkred" if is_hidden else "green"
        comp = coef_all[j] * kval(th - mu)
        nm = ("HIDDEN plateau @ %.4f rad (A=%.2f)" % (mu % np.pi, coef_all[j]) if is_hidden
              else "comp %d @ %.4f rad (A=%.2f)" % (j + 1, mu % np.pi, coef_all[j]))
        fig2.add_trace(go.Scatter(x=thd, y=comp, line=dict(dash="dash", width=1.4 if is_hidden else 1.0, color=col),
                                  name=nm))
        fig2.add_vline(x=mu % np.pi, line=dict(color=col, dash="dot", width=1.4 if is_hidden else 1))
        fig2.add_trace(go.Scatter(x=[mu % np.pi], y=[float(np.interp(mu % np.pi, thd, F))],
                                  mode="markers",
                                  marker=dict(symbol="diamond", size=11 if is_hidden else 9, color=col,
                                              line=dict(width=1, color="black")),
                                  hovertemplate="kernel peak @ %{x:.4f} rad<extra></extra>",
                                  showlegend=False))
    res = F - model_all
    fig2.add_trace(go.Scatter(x=thd, y=res, name="residual", yaxis="y2",
                              line=dict(color="red", width=1.2, dash="dot"),
                              hovertemplate="%{x:.4f} rad<br>resid %{y:.5f}<extra></extra>"))
    if peaks is not None:
        prad = np.mod(peaks[:, 0], np.pi)
        fig2.add_trace(go.Scatter(x=prad, y=np.interp(prad, thd, F), mode="markers",
                                  name="persistence peaks (mod pi)",
                                  marker=dict(symbol="x", size=10, color="red", line=dict(width=2)),
                                  hovertemplate="%{x:.4f} rad<extra></extra>"))
    if gt_rad is not None:
        fig2.add_vline(x=gt_rad % np.pi, line=dict(color="green", dash="dash"),
                       annotation_text="GT %.4f rad" % gt_rad, annotation_position="top right")
    fig2.update_layout(title="Folded correlation + kernel fit (windowed)",
                       height=560,
                       legend=dict(orientation="h", y=1.12, font=dict(size=10)),
                       xaxis=dict(title="rotation angle (rad)", range=[0, np.pi],
                                  tickvals=[0, np.pi / 4, np.pi / 2, 3 * np.pi / 4, np.pi],
                                  ticktext=["0", "pi/4", "pi/2", "3pi/4", "pi"]),
                       yaxis=dict(title="normalized correlation"),
                       yaxis2=dict(title="residual", overlaying="y", side="right", showgrid=False),
                       margin=dict(t=80))
    fig2.show()
else:
    print("Figure 2 skipped (no fit results).")


In [71]:
"""Figure 1 update: kernel-fit peaks (green diamonds = persistence-derived, darkred = HIDDEN plateau)."""
if K is not None and hidden_result is not None and curve is not None:
    for p in kernel_peaks:
        col = "darkred" if p["hidden"] else "green"
        xm = [p["mu"] % (2 * np.pi), (p["mu"] + np.pi) % (2 * np.pi)]
        ym = [float(np.interp(t, x, c)) for t in xm]
        fig1.add_trace(go.Scatter(x=xm, y=ym, mode="markers+text",
                                  name=("kernel HIDDEN @ %.3f rad" % (p["mu"] % np.pi)
                                        if p["hidden"] else "kernel peak @ %.3f rad" % (p["mu"] % np.pi)),
                                  marker=dict(symbol="diamond", size=11 if p["hidden"] else 9, color=col,
                                              line=dict(width=1, color="black")),
                                  text=["%.3f (A=%.2f)" % (p["mu"] % np.pi, p["amp"])] * 2,
                                  textposition="bottom center", textfont=dict(size=9, color=col),
                                  hovertemplate="%{x:.4f} rad<extra></extra>"))
    _redraw()
    print("Kernel-fit peaks added to Figure 1 (diamonds: green = persistence-derived, darkred = hidden plateau).")
else:
    print("Figure 1 update skipped (no fit results).")


Kernel-fit peaks added to Figure 1 (diamonds: green = persistence-derived, darkred = hidden plateau).


In [72]:
"""Combined summary (rad): deduplicated persistence peaks + per-peak kernel scan, GT-free."""
if K is not None:
    print("=== Persistence peaks on [0, pi) ===   (%d unique)" % len(known_all))
    for p in known_all:
        row = peaks[np.argmin(circ_dist(peaks[:, 0] % np.pi, p))]
        print("  %9.4f rad  corr=%.3f  levelPot=%.4f" % (p, row[1], row[3]))
    if hidden_result is not None:
        print("\n=== Per-peak kernel scan (window = anchor +/- %.2f rad, threshold %.1fx, GT-free) ==="
              % (WIN_HALF_RAD, MIN_IMPROV_RATIO))
        print("  %9s %15s %7s %-30s %s" % ("anchor", "hidden mu (rad)", "A", "resid0 -> resid1", "ratio"))
        for r in peak_scan:
            if r["accepted"]:
                print("  %9.4f %15.4f %7.3f %-30s %.1fx"
                      % (r["center"], r["mu"], r["amp"],
                         "%.4f -> %.4f" % (r["resid0"], r["resid1"]), r["ratio"]))
            else:
                print("  %9.4f %15s %7s %-30s below threshold" % (r["center"], "-", "-",
                      "%.4f -> %.4f" % (r["resid0"], r["resid1"])))
        h = hidden_result
        print("\n=== Kernel-fit peaks (best window [%.4f, %.4f] rad around anchor %.4f) ==="
              % (h["lo"], h["hi"], h["center"]))
        print("  %9s %8s %-16s %s" % ("mu (rad)", "A", "source", "type"))
        for p in kernel_peaks:
            print("  %9.4f %8.3f %-16s %s" % (p["mu"] % np.pi, p["amp"],
                  "HIDDEN plateau" if p["hidden"] else "persistence",
                  "shoulder" if p["shoulder"] else "local max"))
        print("  residual %.4f -> %.4f (%.1fx improvement)" % (h["resid0"], h["resid1"], h["ratio"]))
        if gt_rad is not None:
            near = circ_dist(gt_rad % np.pi, h["mu"]) < 0.035
            print("  GT validation (meta row, debug only): GT = %.4f rad -> hidden plateau %s"
                  % (gt_rad, "within 0.035 rad (correct pick)" if near else "NOT near GT (check pair/meta!)"))
        print("\nDifference: the kernel fit confirms the persistence peaks inside the best window")
        print("and adds ONE hidden plateau that classical detection cannot see (shoulder). The scan")
        print("never uses GT - it probes every persistence peak. See the residual trace in Figure 2.")
    else:
        print("\nNo hidden plateau accepted: every window stayed below the %.1fx residual threshold."
              % MIN_IMPROV_RATIO)
else:
    print("Summary skipped (no fit results).")


=== Persistence peaks on [0, pi) ===   (3 unique)
     0.4310 rad  corr=0.861  levelPot=0.7414
     0.9557 rad  corr=0.392  levelPot=0.1538
     2.0739 rad  corr=1.000  levelPot=1.0000

=== Per-peak kernel scan (window = anchor +/- 0.35 rad, threshold 2.0x, GT-free) ===
     anchor hidden mu (rad)       A resid0 -> resid1               ratio
     0.4310          0.5462   0.204 1.7428 -> 0.1829               9.5x
     0.9557               -       - 0.3185 -> 0.1752               below threshold
     2.0739               -       - 0.4875 -> 0.3262               below threshold

=== Kernel-fit peaks (best window [0.0810, 0.7810] rad around anchor 0.4310) ===
   mu (rad)        A source           type
     0.4310    0.609 persistence      local max
     0.9557    0.008 persistence      local max
     0.5462    0.204 HIDDEN plateau   shoulder
  residual 1.7428 -> 0.1829 (9.5x improvement)
  GT validation (meta row, debug only): GT = 5.2360 rad -> hidden plateau NOT near GT (check pair/meta!

## Notes & pitfalls

- **pi-periodicity:** the resampled Fourier magnitude is even in azimuth, so $C(\theta)$ is (bit-)exactly periodic in $\pi$. The folded trace is lossless; angles are physical rotations modulo $\pi$ rad.
- **GT derivation:** `registration_meta.csv` stores the *estimated* rotation and the GT *error* (in degrees); GT = estimated + error, displayed in radians. Verify the pair is the one you ran.
- **GT-free by design:** the scan probes one window per persistence peak; GT only scores the pick (meta row). If the data dir mixes runs (compare file mtimes), the meta row may belong to another pair - the plateau is still found, only the validation line is then meaningless.
- **Gaussian fits will mislead:** the true peak kernel is NOT Gaussian (band-limited correlation: sharp core + tails + side lobes, e.g. a strong lobe near $\pi/2$ rad in one earlier pair). A Gaussian mixture invents spurious components there. Use the kernel from the coefficients.
- **Shoulders are invisible to peak detectors** (local-max / persistence alike): they are not local maxima. Only model fitting (kernel subtraction) reveals them.
- **Resolution limit:** two rotations closer than ~ the kernel core width (roughly 0.05-0.15 rad) cannot be separated; the scan enforces `MIN_SEP_RAD` separation from known components.
- **The kernel is per scan pair:** it is the autocorrelation of the *reference* (scan 2) descriptor. Recompute it for each debug run (the notebook does this automatically).
- Missing files? Run fs2d with debug + `useDirect=true` to get the coefficient files; without them the notebook falls back to an empirical kernel (dominant peak shape).
